# TP — Module 2 : Feature Engineering Spatial
## Application aux enjeux hydrauliques et de dégradation des terres au Burkina Faso

> **Cours** : Analyse Spatiale avec Machine Learning (ASML) — Version 4  
> **Institut** : 2iE — Master Eau, Environnement, Aménagement  
> **Durée** : 4h

---

## Différence avec le CM

| | CM (support de cours) | TP (ce document) |
|-|-----------------------|-----------------|
| **Rôle** | Enseigner les concepts et outils | Appliquer ces outils à de nouveaux problèmes |
| **Variable cible** | Phases IPC (insécurité alimentaire) | Vulnérabilité hydraulique et dégradation des terres |
| **Approche** | Illustrative — code expliqué pas à pas | Analytique — répondre à des questions d'aide à la décision |

> **Les outils sont les mêmes** (GeoPandas, libpysal, rasterstats…)
> **Les questions et les données sont différentes**

---

## Contexte général

Le Programme National d'Alimentation en Eau Potable (PNAEP) du Burkina Faso
doit prioriser ses investissements en infrastructures hydrauliques sur les 45 provinces.
En parallèle, le CILSS signale une accélération de la dégradation des terres dans
le Nord du pays depuis 2015.

**Ce TP répond à deux questions :**

**Application A** : Quelles provinces sont les plus vulnérables en matière d'accès à l'eau ?
→ Construire un indice de vulnérabilité hydraulique à partir de features géospatiales.

**Application B** : Quelles provinces subissent la dégradation végétale la plus préoccupante ?
→ Analyser la tendance NDVI 2020–2023 et son clustering spatial.

---

## Données fournies

| Variable | Description | Source |
|----------|-------------|--------|
| `nb_puits` | Nombre de points d'eau fonctionnels (forages + puits) | DGRE-BF 2023 |
| `taux_acces_eau_pct` | Taux d'accès à l'eau potable (%) | JMP/WHO 2023 |
| `pluvio_mm` | Pluviométrie annuelle moyenne 2018-2022 (mm) | ANAM BF |
| `ndvi_2020` | NDVI moyen en juillet 2020 | Sentinel-2 (Copernicus) |
| `ndvi_2023` | NDVI moyen en juillet 2023 | Sentinel-2 (Copernicus) |
| `pop` | Population 2022 | INSD BF |

## Environnement

In [ ]:
# !pip install geopandas libpysal esda rasterstats statsmodels folium
#             contextily requests scipy scikit-learn --quiet

In [ ]:
import json, io, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
from shapely.geometry import Point, LineString
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial import cKDTree
from libpysal.weights import Queen, KNN
from libpysal.weights import lag_spatial
from esda.moran import Moran, Moran_Local
from statsmodels.stats.outliers_influence import variance_inflation_factor
import folium

TIMEOUT = 15
print('Bibliothèques chargées. GeoPandas :', gpd.__version__)

# ═══════════════════════════════════════════════════════════════
# Données TP embarquées — 20 provinces BF avec données hydrauliques
# et NDVI (variables DIFFÉRENTES du CM qui utilise IPC)
# ═══════════════════════════════════════════════════════════════
FALLBACK_GJ_STR = '{"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"province": "Oudalan", "region": "Sahel", "pop": 322000, "pluvio_mm": 280, "nb_puits": 45, "taux_acces_eau_pct": 28, "ndvi_2020": 0.08, "ndvi_2023": 0.05}, "geometry": {"type": "Polygon", "coordinates": [[[-0.78, 14.23], [-0.18, 14.23], [-0.18, 14.73], [-0.78, 14.73], [-0.78, 14.23]]]}}, {"type": "Feature", "properties": {"province": "Séno", "region": "Sahel", "pop": 325000, "pluvio_mm": 310, "nb_puits": 62, "taux_acces_eau_pct": 35, "ndvi_2020": 0.09, "ndvi_2023": 0.06}, "geometry": {"type": "Polygon", "coordinates": [[[-0.4, 13.85], [0.19999999999999998, 13.85], [0.19999999999999998, 14.35], [-0.4, 14.35], [-0.4, 13.85]]]}}, {"type": "Feature", "properties": {"province": "Soum", "region": "Sahel", "pop": 369000, "pluvio_mm": 350, "nb_puits": 78, "taux_acces_eau_pct": 42, "ndvi_2020": 0.1, "ndvi_2023": 0.07}, "geometry": {"type": "Polygon", "coordinates": [[[-1.85, 13.73], [-1.25, 13.73], [-1.25, 14.23], [-1.85, 14.23], [-1.85, 13.73]]]}}, {"type": "Feature", "properties": {"province": "Yagha", "region": "Sahel", "pop": 214000, "pluvio_mm": 420, "nb_puits": 38, "taux_acces_eau_pct": 30, "ndvi_2020": 0.11, "ndvi_2023": 0.09}, "geometry": {"type": "Polygon", "coordinates": [[[0.45, 13.1], [1.05, 13.1], [1.05, 13.6], [0.45, 13.6], [0.45, 13.1]]]}}, {"type": "Feature", "properties": {"province": "Loroum", "region": "Nord", "pop": 194000, "pluvio_mm": 480, "nb_puits": 55, "taux_acces_eau_pct": 45, "ndvi_2020": 0.14, "ndvi_2023": 0.11}, "geometry": {"type": "Polygon", "coordinates": [[[-2.4, 13.45], [-1.8, 13.45], [-1.8, 13.95], [-2.4, 13.95], [-2.4, 13.45]]]}}, {"type": "Feature", "properties": {"province": "Yatenga", "region": "Nord", "pop": 535000, "pluvio_mm": 520, "nb_puits": 95, "taux_acces_eau_pct": 52, "ndvi_2020": 0.15, "ndvi_2023": 0.13}, "geometry": {"type": "Polygon", "coordinates": [[[-2.65, 13.3], [-2.0500000000000003, 13.3], [-2.0500000000000003, 13.8], [-2.65, 13.8], [-2.65, 13.3]]]}}, {"type": "Feature", "properties": {"province": "Passoré", "region": "Nord", "pop": 313000, "pluvio_mm": 560, "nb_puits": 82, "taux_acces_eau_pct": 55, "ndvi_2020": 0.17, "ndvi_2023": 0.15}, "geometry": {"type": "Polygon", "coordinates": [[[-2.5, 12.64], [-1.9000000000000001, 12.64], [-1.9000000000000001, 13.14], [-2.5, 13.14], [-2.5, 12.64]]]}}, {"type": "Feature", "properties": {"province": "Sanmatenga", "region": "Centre-Nord", "pop": 532000, "pluvio_mm": 590, "nb_puits": 88, "taux_acces_eau_pct": 58, "ndvi_2020": 0.18, "ndvi_2023": 0.16}, "geometry": {"type": "Polygon", "coordinates": [[[-1.5, 12.8], [-0.8999999999999999, 12.8], [-0.8999999999999999, 13.3], [-1.5, 13.3], [-1.5, 12.8]]]}}, {"type": "Feature", "properties": {"province": "Namentenga", "region": "Centre-Nord", "pop": 331000, "pluvio_mm": 620, "nb_puits": 72, "taux_acces_eau_pct": 54, "ndvi_2020": 0.19, "ndvi_2023": 0.18}, "geometry": {"type": "Polygon", "coordinates": [[[-0.8, 12.85], [-0.2, 12.85], [-0.2, 13.35], [-0.8, 13.35], [-0.8, 12.85]]]}}, {"type": "Feature", "properties": {"province": "Bam", "region": "Centre-Nord", "pop": 295000, "pluvio_mm": 640, "nb_puits": 65, "taux_acces_eau_pct": 60, "ndvi_2020": 0.21, "ndvi_2023": 0.19}, "geometry": {"type": "Polygon", "coordinates": [[[-1.7, 13.2], [-1.0999999999999999, 13.2], [-1.0999999999999999, 13.7], [-1.7, 13.7], [-1.7, 13.2]]]}}, {"type": "Feature", "properties": {"province": "Kadiogo", "region": "Centre", "pop": 2415000, "pluvio_mm": 820, "nb_puits": 210, "taux_acces_eau_pct": 88, "ndvi_2020": 0.26, "ndvi_2023": 0.25}, "geometry": {"type": "Polygon", "coordinates": [[[-1.83, 12.11], [-1.23, 12.11], [-1.23, 12.61], [-1.83, 12.61], [-1.83, 12.11]]]}}, {"type": "Feature", "properties": {"province": "Bazega", "region": "Centre-Sud", "pop": 226000, "pluvio_mm": 850, "nb_puits": 58, "taux_acces_eau_pct": 65, "ndvi_2020": 0.29, "ndvi_2023": 0.28}, "geometry": {"type": "Polygon", "coordinates": [[[-1.7, 11.85], [-1.0999999999999999, 11.85], [-1.0999999999999999, 12.35], [-1.7, 12.35], [-1.7, 11.85]]]}}, {"type": "Feature", "properties": {"province": "Zoundweogo", "region": "Centre-Sud", "pop": 244000, "pluvio_mm": 860, "nb_puits": 62, "taux_acces_eau_pct": 67, "ndvi_2020": 0.28, "ndvi_2023": 0.27}, "geometry": {"type": "Polygon", "coordinates": [[[-1.15, 11.6], [-0.55, 11.6], [-0.55, 12.1], [-1.15, 12.1], [-1.15, 11.6]]]}}, {"type": "Feature", "properties": {"province": "Nahouri", "region": "Centre-Sud", "pop": 185000, "pluvio_mm": 870, "nb_puits": 55, "taux_acces_eau_pct": 68, "ndvi_2020": 0.27, "ndvi_2023": 0.26}, "geometry": {"type": "Polygon", "coordinates": [[[-1.4000000000000001, 11.25], [-0.8, 11.25], [-0.8, 11.75], [-1.4000000000000001, 11.75], [-1.4000000000000001, 11.25]]]}}, {"type": "Feature", "properties": {"province": "Boulgou", "region": "Centre-Est", "pop": 398000, "pluvio_mm": 780, "nb_puits": 75, "taux_acces_eau_pct": 56, "ndvi_2020": 0.24, "ndvi_2023": 0.22}, "geometry": {"type": "Polygon", "coordinates": [[[-0.5, 11.6], [0.09999999999999998, 11.6], [0.09999999999999998, 12.1], [-0.5, 12.1], [-0.5, 11.6]]]}}, {"type": "Feature", "properties": {"province": "Ganzourgou", "region": "Plateau Central", "pop": 255000, "pluvio_mm": 750, "nb_puits": 68, "taux_acces_eau_pct": 62, "ndvi_2020": 0.23, "ndvi_2023": 0.21}, "geometry": {"type": "Polygon", "coordinates": [[[-1.05, 12.1], [-0.45, 12.1], [-0.45, 12.6], [-1.05, 12.6], [-1.05, 12.1]]]}}, {"type": "Feature", "properties": {"province": "Houet", "region": "Hauts-Bassins", "pop": 1015000, "pluvio_mm": 980, "nb_puits": 145, "taux_acces_eau_pct": 78, "ndvi_2020": 0.4, "ndvi_2023": 0.38}, "geometry": {"type": "Polygon", "coordinates": [[[-4.6, 10.93], [-4.0, 10.93], [-4.0, 11.43], [-4.6, 11.43], [-4.6, 10.93]]]}}, {"type": "Feature", "properties": {"province": "Comoé", "region": "Cascades", "pop": 274000, "pluvio_mm": 1050, "nb_puits": 82, "taux_acces_eau_pct": 82, "ndvi_2020": 0.46, "ndvi_2023": 0.44}, "geometry": {"type": "Polygon", "coordinates": [[[-4.95, 10.35], [-4.3500000000000005, 10.35], [-4.3500000000000005, 10.85], [-4.95, 10.85], [-4.95, 10.35]]]}}, {"type": "Feature", "properties": {"province": "Poni", "region": "Sud-Ouest", "pop": 267000, "pluvio_mm": 1100, "nb_puits": 78, "taux_acces_eau_pct": 79, "ndvi_2020": 0.5, "ndvi_2023": 0.48}, "geometry": {"type": "Polygon", "coordinates": [[[-3.5999999999999996, 10.05], [-3.0, 10.05], [-3.0, 10.55], [-3.5999999999999996, 10.55], [-3.5999999999999996, 10.05]]]}}, {"type": "Feature", "properties": {"province": "Noumbiel", "region": "Sud-Ouest", "pop": 100000, "pluvio_mm": 1150, "nb_puits": 45, "taux_acces_eau_pct": 80, "ndvi_2020": 0.54, "ndvi_2023": 0.52}, "geometry": {"type": "Polygon", "coordinates": [[[-3.4, 9.75], [-2.8000000000000003, 9.75], [-2.8000000000000003, 10.25], [-3.4, 10.25], [-3.4, 9.75]]]}}]}'

fallback_gdf = gpd.GeoDataFrame.from_features(
    json.loads(FALLBACK_GJ_STR)['features'], crs='EPSG:4326'
)
print(f'Fallback : {len(fallback_gdf)} provinces')
print('Colonnes :', list(fallback_gdf.columns))

# ═══════════════════════════════════════════════════════════════
# Chargement GADM 4.1 — vraies géométries (level-2 = 45 provinces)
# Source : https://gadm.org/download_country.html → GeoJSON level-2
# ═══════════════════════════════════════════════════════════════

URL_GADM_L2 = 'https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_BFA_2.json'

def load_gadm_provinces(fallback):
    try:
        print('Chargement GADM level-2 (45 provinces BF)...')
        r = requests.get(URL_GADM_L2, timeout=TIMEOUT)
        r.raise_for_status()
        gdf_raw = gpd.read_file(io.BytesIO(r.content))
        gdf_raw = gdf_raw.rename(columns={
            'NAME_0':'pays','NAME_1':'region','NAME_2':'province','GID_2':'gid'
        })
        # Joindre les attributs TP depuis le fallback
        attrs = ['province','nb_puits','taux_acces_eau_pct','pluvio_mm',
                 'ndvi_2020','ndvi_2023','pop']
        fb_attrs = fallback[[c for c in attrs if c in fallback.columns]]
        fb_attrs['province_norm'] = fb_attrs['province'].str.lower().str.strip()
        gdf_raw['province_norm'] = gdf_raw['province'].str.lower().str.strip()
        gdf_raw = gdf_raw.merge(fb_attrs.drop(columns='province'),
                                on='province_norm', how='left')
        gdf_raw = gdf_raw.drop(columns='province_norm')
        print(f'[GADM ✅] {len(gdf_raw)} provinces | CRS: {gdf_raw.crs}')
        return gdf_raw, 'GADM 4.1'
    except Exception as e:
        warnings.warn(f'GADM inaccessible ({e}) → fallback', stacklevel=2)
        print('[FALLBACK ⚠️] Données embarquées utilisées')
        return fallback.copy(), 'fallback'

gdf, source_gdf = load_gadm_provinces(fallback_gdf)
print(f'\nSource : {source_gdf} | {len(gdf)} provinces')

---
# Application A — Indice de Vulnérabilité Hydraulique

## Contexte de la mission

La DGRE (Direction Générale des Ressources en Eau) du BF doit allouer
un budget limité pour la construction de nouveaux forages.
Elle vous demande de construire un **indice composite de vulnérabilité hydraulique**
qui combine la forme des provinces, la densité de population non desservie,
et la structure spatiale du déficit d'accès à l'eau.

**Critères retenus par la DGRE :**
- Provinces avec fort déficit d'accès (taux < 50%)
- Provinces en cluster de vulnérabilité (entourées d'autres provinces déficitaires)
- Provinces avec forte densité de population non desservie (pression sur les points d'eau existants)

---
## Exercice A.1 — Features géométriques et hydrauliques (2 pts)

L'accessibilité interne d'une province détermine combien de forages il faut pour la couvrir.
Une province allongée et peu compacte nécessite plus de points d'eau dispersés qu'une province compacte.

1. Reprojetez en **EPSG:32630** et calculez :
   - `superficie_km2`, `compacite` (Polsby-Popper)
   - `densite_pop` (pop / superficie_km2)
2. Calculez `puits_par_10000hab` = (nb_puits / pop) × 10 000
   - C'est l'indicateur DGRE standard de couverture en points d'eau
3. Calculez `pop_non_desservie` = pop × (1 - taux_acces_eau_pct / 100)
4. Tracez une carte de `taux_acces_eau_pct` avec une échelle de couleur
   rouge (< 40%) → jaune (50–65%) → vert (> 75%)

> 💡 La compacité n'est pas dans les données DGRE — c'est vous qui la calculez
> à partir des géométries GADM. C'est un apport direct de la méthode géospatiale.

In [ ]:
# ── A.1 : Votre code ici ──

# 1. Features géométriques


# 2. Couverture en points d'eau


# 3. Population non desservie


# 4. Carte taux d'accès
fig, ax = plt.subplots(figsize=(10, 8))
# ...
plt.savefig('TP_A1_acces_eau.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Exercice A.2 — Voisinage hydraulique et clustering (3 pts)

Les problèmes d'accès à l'eau sont rarement isolés : une province déficitaire est
souvent entourée de provinces également déficitaires (même aquifère, même géologie,
même pluviométrie). Cette structure spatiale est cruciale pour planifier les
interconnexions de réseaux d'adduction.

1. Construisez la matrice W Queen sur les géométries disponibles
2. Calculez le **Moran I** de `taux_acces_eau_pct`
   - Y a-t-il un clustering spatial du déficit d'accès à l'eau ?
   - Comparez avec le Moran I d'IPC du CM : lequel est plus fort ?
3. Calculez `acces_lag` = lag spatial de `taux_acces_eau_pct`
4. Calculez les LISA → `acces_lisa_cat`
   - HH = cluster de faible accès = zone d'intervention prioritaire
   - LL = cluster de bon accès = à maintenir
5. Tracez la carte LISA des déficits hydrauliques

> 📊 **Question analytique** : comparez la carte LISA hydraulique avec la
> carte IPC du CM. Les zones de faible accès à l'eau (HH hydraulique)
> correspondent-elles aux zones de forte insécurité alimentaire (HH IPC) ?
> Justifiez en 2-3 lignes.

In [ ]:
# ── A.2 : Votre code ici ──

# 1. Matrice W


# 2. Moran I de l'accès à l'eau


# 3. Lag spatial


# 4. LISA


# 5. Carte LISA
fig, ax = plt.subplots(figsize=(10, 8))
# ...
plt.savefig('TP_A2_lisa_eau.png', dpi=150, bbox_inches='tight')
plt.show()

# Réponse analytique (commentaire) :
# ...

---
## Exercice A.3 — Indice composite et priorisation (3 pts)

L'objectif final est de produire un **classement des provinces** selon leur
vulnérabilité hydraulique composite, pour guider la DGRE dans ses investissements.

1. Construisez un indice composite `IVH` (Indice de Vulnérabilité Hydraulique) :
   - Normalisez chaque composante en [0,1] avec MinMaxScaler
   - `IVH = 0.40 × (1 - taux_acces_eau_pct_norm)`
     `+ 0.30 × (1 - puits_par_10000hab_norm)`
     `+ 0.20 × (1 - acces_lag_norm)` *(pression de voisinage)*
     `+ 0.10 × compacite_inv_norm` *(1/compacite normalisée → difficile à couvrir)*
2. Identifiez le **top-5 des provinces prioritaires** (IVH le plus élevé)
   - Affichez : nom, région, IVH, taux_acces_eau_pct, acces_lisa_cat
3. Calculez le VIF sur les 4 composantes de l'IVH
   - Y a-t-il une redondance entre `taux_acces_eau_pct` et `acces_lag` ?
4. Produisez une carte Folium interactive avec :
   - Choroplèthe de l'IVH
   - Popup : nom province, IVH, taux d'accès, catégorie LISA

> 💡 Ce type d'indice composite est utilisé par l'UNICEF et la Banque Mondiale
> pour prioriser les investissements WASH (Water, Sanitation, Hygiene) en Afrique.

In [ ]:
# ── A.3 : Votre code ici ──

# 1. Indice composite IVH


# 2. Top-5 provinces prioritaires


# 3. VIF des composantes


# 4. Carte Folium
m = folium.Map(location=[12.37, -1.54], zoom_start=6)
# ...
m.save('TP_A3_IVH_interactive.html')
print('Carte sauvegardée.')

---
# Application B — Surveillance de la Dégradation des Terres

## Contexte de la mission

Le CILSS (Comité permanent Inter-États de Lutte contre la Sécheresse au Sahel)
a signalé une accélération de la dégradation des terres dans le Nord-BF depuis 2015.
Il vous demande une analyse spatiale du changement de végétation entre 2020 et 2023
pour identifier les zones critiques et leur structure spatiale.

**Les données disponibles :**
- `ndvi_2020` : NDVI moyen juillet 2020 (référence pré-dégradation)
- `ndvi_2023` : NDVI moyen juillet 2023 (situation récente)

---
## Exercice B.1 — Analyse du changement de végétation (2 pts)

1. Calculez `delta_ndvi` = ndvi_2023 − ndvi_2020
   - Valeur négative = régression de la végétation
   - Valeur positive = amélioration (reboisement, pluies favorables)
2. Calculez `taux_changement_pct` = (delta_ndvi / ndvi_2020) × 100
3. Classez chaque province en :
   `'Dégradation sévère'` si taux < -15%
   `'Dégradation modérée'` si -15% ≤ taux < -5%
   `'Stable'` si -5% ≤ taux < +5%
   `'Amélioration'` si taux ≥ +5%
4. Tracez un graphique à barres horizontales trié par `delta_ndvi`
   (du plus dégradé au plus amélioré), coloré par classe

> ⚠️ Attention : une province avec un ndvi_2020 très faible (Sahel aride)
> aura un delta_ndvi faible en valeur absolue mais un taux_changement élevé.
> Utiliser le taux plutôt que la différence absolue pour comparer justement
> des provinces à végétation de base très différente.

In [ ]:
# ── B.1 : Votre code ici ──

# 1. Calcul du delta NDVI


# 2. Taux de changement en %


# 3. Classification du changement


# 4. Graphique à barres
fig, ax = plt.subplots(figsize=(10, 8))
# ...
plt.savefig('TP_B1_delta_ndvi.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Exercice B.2 — Structure spatiale de la dégradation (2 pts)

La dégradation des terres n'est pas aléatoire : elle se propage par des
mécanismes physiques (érosion éolienne, sécheresse régionale, overgrazing)
et humains (déforestation, flux de réfugiés qui surexploitent les ressources).
Analyser sa structure spatiale permet d'identifier les fronts d'avancée.

1. Calculez le **Moran I** de `delta_ndvi`
   - La dégradation est-elle spatialement structurée ou aléatoire ?
2. Calculez `delta_ndvi_lag` = lag spatial de `delta_ndvi`
   - Interprétez : une province avec delta_ndvi=-0.03 et delta_ndvi_lag=-0.02
     vs delta_ndvi=-0.03 et delta_ndvi_lag=+0.01. Laquelle est la plus préoccupante ?
3. Calculez les LISA de `delta_ndvi`
   - `HH` ici = cluster de provinces qui S'AMÉLIORENT toutes
   - `LL` ici = cluster de provinces qui SE DÉGRADENT toutes → front de dégradation
4. Tracez : carte LISA dégradation à gauche, carte du delta_ndvi à droite

In [ ]:
# ── B.2 : Votre code ici ──

# 1. Moran I de delta_ndvi


# 2. Lag spatial

# Interprétation (commentaire) :
# ...

# 3. LISA


# 4. Cartes
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
# ...
plt.savefig('TP_B2_spatial_degradation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Exercice B.3 — Rapport d'alerte précoce (3 pts)

Le CILSS vous demande de produire un **rapport synthétique** identifiant les
provinces à surveiller en priorité, avec les features permettant de l'expliquer.

1. Créez une `feature_matrix_degradation` avec :
   - `delta_ndvi`, `taux_changement_pct`
   - `delta_ndvi_lag` (contexte régional de dégradation)
   - `pluvio_mm` (facteur climatique)
   - `densite_pop` (pression anthropique)
   - `superficie_km2` (taille du territoire affecté)
2. Calculez le VIF — y a-t-il une redondance entre `delta_ndvi` et `delta_ndvi_lag` ?
3. Produisez le **top-5 des provinces en alerte** :
   - Critère : combinaison de `taux_changement_pct` le plus négatif
     ET `delta_ndvi_lag` également négatif (front de dégradation régionale)
4. Pour ces 5 provinces, rédigez en commentaire :
   - La raison principale probable de la dégradation (pluvio, densité, position géographique)
   - Une recommandation d'intervention (reboisement, gestion pastorale, etc.)

> 📊 **Question de synthèse** : en comparant vos résultats Application A et Application B,
> identifiez 2 provinces qui cumulent à la fois une forte vulnérabilité hydraulique (IVH élevé)
> ET une forte dégradation végétale. Ces provinces représentent la double crise eau+terres —
> la situation la plus préoccupante pour la sécurité alimentaire future.

In [ ]:
# ── B.3 : Votre code ici ──

# 1. Feature matrix dégradation


# 2. VIF


# 3. Top-5 provinces en alerte


# 4. Analyse qualitative (commentaires)
# Province 1 :
# Province 2 :
# Province 3 :
# Province 4 :
# Province 5 :

# Question synthèse — provinces double crise eau + terres :
# ...

---
## Conclusion du TP

| Application | Outils utilisés (du CM) | Question répondue |
|------------|-------------------------|------------------|
| A.1 | Géométrie, compacité, rasterstats | Quelles provinces sont difficiles à équiper en eau ? |
| A.2 | Matrice W, Moran I, LISA | Le déficit hydraulique est-il spatialement structuré ? |
| A.3 | Indice composite, VIF, Folium | Comment prioriser les investissements WASH ? |
| B.1 | Attributs, classification | Où la végétation se dégrade-t-elle le plus ? |
| B.2 | Lag spatial, LISA | Y a-t-il un front de dégradation spatiale ? |
| B.3 | Feature matrix, VIF | Quelles provinces cumulent tous les risques ? |

---
**Module 3** : ces feature matrices (Application A et B) pourront être combinées
avec la feature matrix IPC du CM pour entraîner un classifieur multi-risques.